# Training Qwen3.6-35B-A3B on DAPO-math

This tutorial trains **Qwen3.6-35B-A3B** (a 35B-parameter MoE model
with ~3B active) on grade-school math problems from
[DAPO-math-17k](https://huggingface.co/datasets/zhuzilin/dapo-math-17k).

The loop:
1. Load math problems from HuggingFace via `HuggingFaceDataset`.
2. Score model outputs using slime's built-in `deepscaler` reward
   model, which extracts the final numerical answer and compares
   it to the ground truth.
3. Feed that score back as a GRPO reward through SLIME.
4. Compare base vs. trained accuracy.

Qwen3.6-35B-A3B uses slime's mbridge conversion path:
the HuggingFace checkpoint is pre-converted to torch_dist format
before training, enabling fast batched weight sync during training steps.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

# Skip if modal_training_gym is already importable (e.g. a local editable
# checkout) so your edits keep taking effect and the env stays synced.
if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
from modal_training_gym import (
    Endpoint,
    HuggingFaceDataset,
    Qwen3_6_35B,
    TrainConfig,
    list_checkpoints,
)
from modal_training_gym.train_recipes.slime_recipe import Qwen3_6_35b_Recipe

## Load DAPO-math from HuggingFace

[DAPO-math-17k](https://huggingface.co/datasets/zhuzilin/dapo-math-17k)
contains ~17k math problems with ground-truth answers. We use a
small subset for this tutorial — 100 training samples and 20 for eval.

In [ ]:
class MathDataset(HuggingFaceDataset):
    hf_repo = "zhuzilin/dapo-math-17k"
    input_column = "prompt"
    output_column = "label"
    output_format = "jsonl"
    apply_chat_template = True

dataset = MathDataset(n_rows=120)

Let's take a quick look at the dataset.

In [ ]:
rows = dataset.load()
for row in rows.select(range(2)):
    prompt = row["prompt"]
    if isinstance(prompt, list):
        prompt = prompt[0]["content"] if prompt else ""
    print(prompt[:200])
    print(f"  label: {row['label']}")
    print()

## Train with SLIME

This MoE model runs on 1 × 8×H100 with TP2, PP2, CP1, EP4,
and optimizer CPU offload, matching the native Slime parallelism
that works for Qwen3.6-35B-A3B.

Key points:
- **`rm_type="deepscaler"`** — slime's built-in math reward that
  extracts and compares numerical answers. No custom reward function
  or sandbox needed.
- The HF checkpoint is pre-converted to torch_dist format; slime's
  implicit mbridge mode handles fast weight sync during training steps.
- Built-in slime model args come from
  `scripts/models/qwen3.5-35B-A3B.sh`; the tutorial does not patch slime.

In [ ]:
model = Qwen3_6_35B()
training_run = TrainConfig(
    model=model,
    dataset=dataset,
    recipe=Qwen3_6_35b_Recipe(
        rm_type="deepscaler",
        num_rollout=10,
    ),
)
print("Starting training...")
train_result = training_run.train()
print(f"Training run id: {train_result.training_run_id}")

## Serve the trained model

`Endpoint.launch` provisions a Modal endpoint that mounts the
checkpoint volume and serves the weights behind an OpenAI-compatible
API. Slime Megatron checkpoints are converted to Hugging Face format
during launch. The endpoint name is derived from the model and checkpoint.

`launch` returns as soon as the endpoint has a URL; loading a 35B MoE
checkpoint off the volume takes considerably longer than that, which
is what `wait_until_ready` waits for.

In [ ]:
checkpoint = list_checkpoints(train_result.training_run_id)[-1]
endpoint = Endpoint.launch(model, checkpoint, unauthenticated=True)
endpoint.wait_until_ready(timeout=45 * 60)
print(f"Trained model URL: {endpoint.url}")